In [ ]:
# %% [markdown]
# # Multi‑Language Translation App – Experiments
#
# This notebook tests all core functionalities before integrating into the Streamlit app.

# %% [markdown]
# ## 1. Setup & Dependencies
#
# Install required packages if not already installed:
# ```
# !pip install googletrans==4.0.0-rc1 gTTS speechrecognition pyaudio pytesseract pillow python-docx PyPDF2
# ```
# Also ensure Tesseract OCR is installed on your system (e.g., `sudo apt install tesseract-ocr` on Linux).

# %%
import googletrans
from googletrans import Translator
from gtts import gTTS
import speech_recognition as sr
import pytesseract
from PIL import Image
import PyPDF2
import docx
import json
import os
from datetime import datetime

# %%
print("All imports successful!")

# %% [markdown]
# ## 2. Text Translation (googletrans)

# %%
translator = Translator()
text = "Hello, how are you?"
result = translator.translate(text, src='en', dest='es')
print(f"Original: {text}")
print(f"Translated: {result.text}")

# %% [markdown]
# ## 3. Text‑to‑Speech (gTTS)

# %%
tts = gTTS("Hello, this is a test.", lang='en')
tts.save("test_audio.mp3")
print("Audio saved as test_audio.mp3")
# To play in notebook:
# from IPython.display import Audio
# Audio("test_audio.mp3")

# %% [markdown]
# ## 4. Speech‑to‑Text (speech_recognition)
#
# **Note:** This will use your microphone. Uncomment and run the cell below.

# %%
# recognizer = sr.Recognizer()
# with sr.Microphone() as source:
#     print("Say something...")
#     audio = recognizer.listen(source, timeout=5)
#     try:
#         text = recognizer.recognize_google(audio, language='en-US')
#         print(f"You said: {text}")
#     except sr.UnknownValueError:
#         print("Could not understand")
#     except sr.RequestError as e:
#         print(f"Error: {e}")

# %% [markdown]
# ## 5. OCR with Tesseract (Image to Text)

# %%
# Load an image (you can replace with your own image path)
image_path = "sample_image.png"  # Change to an existing image
if os.path.exists(image_path):
    img = Image.open(image_path)
    text = pytesseract.image_to_string(img, lang='eng')
    print("Extracted text:")
    print(text)
else:
    print("Please place a sample image as 'sample_image.png' or update the path.")

# %% [markdown]
# ## 6. Document Extraction

# ### 6.1 PDF extraction

# %%
pdf_path = "sample.pdf"
if os.path.exists(pdf_path):
    with open(pdf_path, 'rb') as f:
        reader = PyPDF2.PdfReader(f)
        pdf_text = ""
        for page in reader.pages:
            pdf_text += page.extract_text() or ""
        print("PDF content (first 500 chars):")
        print(pdf_text[:500])
else:
    print("No PDF found, skipping.")

# ### 6.2 DOCX extraction

# %%
docx_path = "sample.docx"
if os.path.exists(docx_path):
    doc = docx.Document(docx_path)
    doc_text = "\n".join([p.text for p in doc.paragraphs])
    print("DOCX content (first 500 chars):")
    print(doc_text[:500])
else:
    print("No DOCX found, skipping.")

# ### 6.3 TXT extraction

# %%
txt_path = "sample.txt"
if os.path.exists(txt_path):
    with open(txt_path, 'r', encoding='utf-8') as f:
        txt_text = f.read()
        print("TXT content (first 500 chars):")
        print(txt_text[:500])
else:
    print("No TXT found, skipping.")

# %% [markdown]
# ## 7. Translation of Extracted Text (Document)
#
# Combine extraction with translation.

# %%
if os.path.exists(txt_path):
    with open(txt_path, 'r', encoding='utf-8') as f:
        content = f.read()
    # Translate to Spanish
    result = translator.translate(content, src='en', dest='es')
    print("Translated TXT (first 500 chars):")
    print(result.text[:500])

# %% [markdown]
# ## 8. History Management (JSON)
#
# Save and load translation history.

# %%
history_file = "history.json"

# Sample history entry
entry = {
    "timestamp": datetime.now().isoformat(),
    "type": "Text",
    "source": "Hello",
    "translation": "Hola",
    "src_lang": "English",
    "tgt_lang": "Spanish"
}

# Load existing history if file exists
if os.path.exists(history_file):
    with open(history_file, 'r', encoding='utf-8') as f:
        history = json.load(f)
else:
    history = []

# Append and save
history.append(entry)
with open(history_file, 'w', encoding='utf-8') as f:
    json.dump(history, f, indent=2, ensure_ascii=False)

print(f"Saved {len(history)} entries.")
print("Last entry:", history[-1])

# %% [markdown]
# ## 9. Language List
#
# The same language dictionary used in the app.

# %%
def get_languages():
    return {
        'en': 'English',
        'es': 'Spanish',
        'fr': 'French',
        'de': 'German',
        'it': 'Italian',
        'pt': 'Portuguese',
        'ru': 'Russian',
        'ja': 'Japanese',
        'ko': 'Korean',
        'zh-cn': 'Chinese (Simplified)',
        'ar': 'Arabic',
        'hi': 'Hindi',
    }

langs = get_languages()
print("Available languages:", list(langs.values()))

# %% [markdown]
# ## 10. Next Steps
#
# - Integrate all these pieces into the Streamlit pages.
# - Handle errors gracefully.
# - Consider API limits (googletrans may have rate limiting).
# - For production, consider using official cloud translation APIs.